# Momentum, RMSprop, and Adam

> Based on Stanford CS230 — C2M2, Andrew Ng / DeepLearning.AI

Vanilla gradient descent oscillates in directions with high curvature and moves slowly in flat directions. Momentum, RMSprop, and Adam each address this differently — Adam combines the best of both.

## Learning Objectives

1. Implement gradient descent with momentum using exponentially weighted averages
2. Implement RMSprop and understand per-parameter adaptive step sizes
3. Implement Adam and explain its bias-corrected moment estimates
4. Visually compare all optimisers on a contour landscape

## Formulas

### Momentum
$$V_{dW} = \beta_1 V_{dW} + (1-\beta_1)\,dW \qquad W \leftarrow W - \alpha V_{dW}$$

### RMSprop
$$S_{dW} = \beta_2 S_{dW} + (1-\beta_2)\,dW^2 \qquad W \leftarrow W - \alpha \frac{dW}{\sqrt{S_{dW}} + \varepsilon}$$

### Adam (Adaptive Moment Estimation)
$$V_{dW}^{\text{corr}} = \frac{V_{dW}}{1-\beta_1^t}, \quad S_{dW}^{\text{corr}} = \frac{S_{dW}}{1-\beta_2^t}$$
$$W \leftarrow W - \alpha \frac{V_{dW}^{\text{corr}}}{\sqrt{S_{dW}^{\text{corr}}} + \varepsilon}$$

**Typical hyperparameters:** $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\varepsilon = 10^{-8}$, $\alpha$ = tuned.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- Toy loss surface: elongated bowl ----
# f(w1, w2) = w1^2 / 10 + w2^2
def f(w):  return w[0]**2 / 10 + w[1]**2
def df(w): return np.array([w[0] / 5, 2*w[1]])

def run_optimizer(name, w0, n_steps, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    w = w0.copy()
    V = np.zeros(2); S = np.zeros(2); t = 0
    path = [w.copy()]
    for _ in range(n_steps):
        t += 1
        g = df(w)
        if name == 'GD':
            w -= lr * g
        elif name == 'Momentum':
            V = beta1*V + (1-beta1)*g
            w -= lr * V
        elif name == 'RMSprop':
            S = beta2*S + (1-beta2)*g**2
            w -= lr * g / (np.sqrt(S) + eps)
        elif name == 'Adam':
            V = beta1*V + (1-beta1)*g
            S = beta2*S + (1-beta2)*g**2
            Vc = V / (1 - beta1**t)
            Sc = S / (1 - beta2**t)
            w -= lr * Vc / (np.sqrt(Sc) + eps)
        path.append(w.copy())
    return np.array(path)

w0 = np.array([8.0, 3.0])
n_steps = 80
configs = [
    ('GD',       0.5,  P[2]),
    ('Momentum', 0.5,  P[0]),
    ('RMSprop',  0.3,  P[3]),
    ('Adam',     0.5,  P[1]),
]

# Contour grid
w1_range = np.linspace(-10, 10, 400)
w2_range = np.linspace(-4, 4, 400)
W1, W2 = np.meshgrid(w1_range, w2_range)
Z = W1**2 / 10 + W2**2

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Optimiser Paths on Elongated Loss Surface  f = w₁²/10 + w₂²', fontsize=13, fontweight='bold')

for ax, (name, lr, color) in zip(axes.flatten(), configs):
    path = run_optimizer(name, w0, n_steps, lr)
    costs = [f(p) for p in path]
    ax.contour(W1, W2, Z, levels=np.logspace(-1, 2, 25), cmap='Blues', alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], color=color, lw=2, marker='o', ms=2.5, zorder=3)
    ax.plot(*w0,   'o', color='#888', ms=9, zorder=4, label='Start')
    ax.plot(0, 0, '*', color=P[2], ms=14, zorder=5, label='Optimum')
    ax.set_title(f'{name}  (lr={lr})   steps to <0.01: '
                 f'{next((i for i,c in enumerate(costs) if c < 0.01), ">80")}', fontsize=10)
    ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.show()

# ---- Cost curves ----
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.set_title('Cost Curves — All Optimisers', fontsize=12)
for name, lr, color in configs:
    path = run_optimizer(name, w0, n_steps, lr)
    costs = [f(p) for p in path]
    ax.semilogy(costs, color=color, lw=2.2, label=f'{name}  (lr={lr})')
ax.set_xlabel('Step')
ax.set_ylabel('Loss (log scale)')
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


## Intuition — What Each Term Does in Adam

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 640 180" width="640" height="180" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Formula boxes -->
  <rect x="20"  y="30" width="180" height="130" rx="6" fill="#5b9bd5" fill-opacity="0.10" stroke="#5b9bd5" stroke-width="1.5"/>
  <rect x="230" y="30" width="180" height="130" rx="6" fill="#e05c5c" fill-opacity="0.10" stroke="#e05c5c" stroke-width="1.5"/>
  <rect x="440" y="30" width="180" height="130" rx="6" fill="#7ecba1" fill-opacity="0.10" stroke="#7ecba1" stroke-width="1.5"/>
  <!-- Titles -->
  <text x="110" y="52" text-anchor="middle" fill="#3a7bbf" font-size="11" font-weight="bold">1st moment  V</text>
  <text x="110" y="68" text-anchor="middle" fill="#3a7bbf" font-size="10">(Momentum)</text>
  <text x="320" y="52" text-anchor="middle" fill="#b03a3a" font-size="11" font-weight="bold">2nd moment  S</text>
  <text x="320" y="68" text-anchor="middle" fill="#b03a3a" font-size="10">(RMSprop)</text>
  <text x="530" y="52" text-anchor="middle" fill="#3d7a5e" font-size="11" font-weight="bold">Combined Update</text>
  <!-- Content -->
  <text x="30"  y="90"  fill="#555" font-size="10">EWA of gradients</text>
  <text x="30"  y="107" fill="#555" font-size="10">Smooth oscillations</text>
  <text x="30"  y="124" fill="#555" font-size="10">Accelerates in</text>
  <text x="30"  y="139" fill="#555" font-size="10">consistent directions</text>
  <text x="240" y="90"  fill="#555" font-size="10">EWA of grad²</text>
  <text x="240" y="107" fill="#555" font-size="10">Adapts step per param</text>
  <text x="240" y="124" fill="#555" font-size="10">Large S → small step</text>
  <text x="240" y="139" fill="#555" font-size="10">Small S → large step</text>
  <text x="450" y="90"  fill="#555" font-size="10">W ← W − α V/√S</text>
  <text x="450" y="107" fill="#555" font-size="10">Fast + adaptive</text>
  <text x="450" y="124" fill="#555" font-size="10">Bias correction fixes</text>
  <text x="450" y="139" fill="#555" font-size="10">early underestimate</text>
  <!-- Arrows -->
  <line x1="200" y1="95" x2="227" y2="95" stroke="#888" stroke-width="1.3"/>
  <polygon points="228,95 220,91 220,99" fill="#888"/>
  <line x1="410" y1="95" x2="437" y2="95" stroke="#888" stroke-width="1.3"/>
  <polygon points="438,95 430,91 430,99" fill="#888"/>
  <!-- β labels on arrows -->
  <text x="213" y="89" text-anchor="middle" fill="#888" font-size="9">+</text>
  <text x="423" y="89" text-anchor="middle" fill="#888" font-size="9">÷</text>
</svg>
